# Build Gold

Runs every `.sql` file in `gold/sql/` in filename order.

That is the whole dependency mechanism: the numbers say dimensions first, then
facts, then views. No topological sort, no config — if you add a product, you name
the file so it sorts after whatever it reads.

Each file contains one complete statement (`CREATE OR REPLACE TABLE|VIEW …`), so
this notebook has no idea whether it is building a table or a view — it just runs
SQL. Files ending `.optional.sql` may fail without failing the job (metric views
are a preview feature and are not enabled in every workspace).

Gold is a **full rebuild** every run. It holds nothing that cannot be recomputed
from Silver, which is what makes that safe.

In [ ]:
import sys
from datetime import date
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "common_utils").is_dir():
        sys.path.insert(0, str(candidate))
        break

from common_utils.logger import get_logger, log_info, log_warning
from common_utils.observability import ensure_ops_schema, new_run_id, track
from common_utils.settings import parse_run_date, project_root
from common_utils.writers import create_namespace

In [ ]:
dbutils.widgets.text("sql_folder", "gold/sql")
dbutils.widgets.text("only", "")  # optional comma-separated file stems, e.g. "05_fact_sales_order_line"
dbutils.widgets.text("catalog", "retaildataplatform")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("gold_schema", "gold")
dbutils.widgets.text("run_date", date.today().isoformat())

catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")
run_date = parse_run_date(dbutils.widgets.get("run_date"))
run_id = new_run_id()
logger = get_logger("gold")

folder = project_root() / dbutils.widgets.get("sql_folder")
only = {s.strip() for s in dbutils.widgets.get("only").split(",") if s.strip()}
files = [p for p in sorted(folder.glob("*.sql")) if not only or p.stem.replace(".optional", "") in only]
print("will run:", [p.name for p in files])

In [ ]:
def render(sql: str) -> str:
    """Substitute the environment placeholders so the same SQL deploys to any catalog."""
    return sql.replace("${catalog}", catalog).replace("${silver}", silver_schema).replace("${gold}", gold_schema)


create_namespace(spark, catalog, gold_schema, comment="Gold: star schema, serving views and governed metrics for BI and AI")
ensure_ops_schema(spark, catalog)

for path in files:
    name = path.stem.replace(".optional", "")
    optional = path.name.endswith(".optional.sql")
    statement = render(path.read_text())
    try:
        with track(spark, catalog, run_id, run_date, task="build_gold", layer="gold", entity=name) as stats:
            spark.sql(statement)
            try:
                stats.rows_written = spark.table(f"{catalog}.{gold_schema}.{name.split('_', 1)[1]}").count()
            except Exception:  # noqa: BLE001 - row count is nice to have, not worth failing for
                stats.rows_written = None
            log_info(logger, "built", product=name, optional=optional)
    except Exception as exc:  # noqa: BLE001
        if not optional:
            raise
        log_warning(logger, "optional product skipped", product=name, error=str(exc).splitlines()[0][:200])

## Reconciliation
The line total, the header total and the Silver total must agree. If they do not,
the star is wrong and no dashboard built on it can be trusted.

In [ ]:
display(
    spark.sql(
        f"""
        SELECT
          (SELECT sum(net_amount)   FROM {catalog}.{gold_schema}.fact_sales_order_line) AS lines_net,
          (SELECT sum(net_amount)   FROM {catalog}.{gold_schema}.fact_sales_order)      AS headers_net,
          (SELECT sum(gross_amount) FROM {catalog}.{gold_schema}.fact_sales_order_line) AS lines_gross,
          (SELECT sum(order_gross_amount) FROM {catalog}.{silver_schema}.sales_orders WHERE is_current) AS silver_gross,
          (SELECT count(*) FROM {catalog}.{gold_schema}.fact_sales_order_line WHERE customer_sk = -1)   AS unknown_customer_lines
        """
    )
)